# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
if hasattr(metadata, 'keywords'):
    print(f"Keywords: {', '.join(metadata.keywords)}")
if hasattr(metadata, 'datePublished'):
    print(f"Date Published: {metadata.datePublished}")
if hasattr(metadata, 'license'):
    print(f"License: {metadata.license}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
from pprint import pprint

# Helper: Get all record sets and print their IDs and fields
print("Record sets in the dataset (referenced by their '@id'):")
if hasattr(metadata, 'recordSets') and metadata.recordSets:
    record_sets = metadata.recordSets
elif hasattr(metadata, 'recordSet') and metadata.recordSet:
    # Some Croissant schemas use 'recordSet' instead of 'recordSets'
    record_sets = metadata.recordSet
else:
    record_sets = []

record_set_ids = []  # to fill for use in next section
all_fields_info = dict()

for rs in record_sets:
    rset_id = rs.get('@id', None)
    record_set_ids.append(rset_id)
    rset_name = rs.get('name', '')
    print(f"- Record Set name: '{rset_name}', @id: '{rset_id}'")
    print("  Fields and columns:")
    field_info = dict()
    # Fields might be a list of dicts
    fields = rs.get('fields', [])
    for field in fields:
        field_id = field.get('@id', '')
        field_name = field.get('name', '')
        col_ids = []
        # Each field can have columns (list of dicts with @id)
        if 'columns' in field and field['columns']:
            for col in field['columns']:
                col_id = col.get('@id', '')
                col_ids.append(col_id)
        print(f"    - Field: '{field_name}', @id: '{field_id}', Columns: {col_ids}")
        field_info[field_id] = {
            'name': field_name,
            'columns': col_ids,
            'dataType': field.get('dataType', None)
        }
    all_fields_info[rset_id] = field_info
if not record_set_ids:
    print("No record sets found in the metadata.")
else:
    print(f"\nAvailable record sets by @id: {record_set_ids}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
# Use the discovered '@id's in record_set_ids
dfs = dict()

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dfs[record_set_id] = df
        print(f"Loaded record set {record_set_id} with columns: {df.columns.tolist()}")
        print(df.head(3))
    except Exception as e:
        print(f"Failed to load record set {record_set_id}: {e}")
        dfs[record_set_id] = None

# Example: Use the first record set for further analysis
if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"\nMain record set for exploration: {main_rs_id}")
    print(f"Columns: {dfs[main_rs_id].columns.tolist()}")
    display(dfs[main_rs_id].head())
else:
    print("No record sets available for extraction.")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example EDA on the main record set (use actual field @ids from the earlier overview):

import numpy as np
pd.set_option('display.max_columns', 20)

if record_set_ids:
    # Pick a numeric field: adjust below using a known numeric field @id from the specific dataset
    rs_id = main_rs_id
    df = dfs[rs_id]
    # Try to auto-select a numeric field
    # Populate field map with @id as column name
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_fields:
        # Try to cast any field that looks numeric
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except Exception:
                pass
        numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Using numeric field: {numeric_field}")
        # Filtering
        threshold = df[numeric_field].quantile(0.75)  # filter e.g., upper quartile
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold} (upper quartile): {len(filtered_df)} rows")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical field
        group_fields = [col for col in df.columns if pd.api.types.is_object_dtype(df[col]) and col!=numeric_field]
        if group_fields:
            group_field = group_fields[0]
            print(f"\nGrouped data by '{group_field}' (mean of numeric fields):")
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(grouped_df.head())
        else:
            print("No categorical field available for grouping.")
        filtered_df.reset_index(drop=True, inplace=True)
    else:
        print('No numeric fields found for exploration.')
else:
    print('No main record set loaded for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram and boxplot for the numeric field if available
if record_set_ids and numeric_fields:
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field}")

    plt.subplot(1,2,2)
    sns.boxplot(x=df[numeric_field].dropna())
    plt.title(f"Boxplot of {numeric_field}")
    plt.show()

    # If grouping is possible
    if group_fields:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No record set or numeric field available for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

**In this notebook, we've loaded and explored the FAIR² dataset, extracted its metadata, listed the available record sets and fields by their `@id`, and performed basic exploratory data analysis and visualization using the `mlcroissant` library. You can further adapt this notebook for more in-depth analysis or modeling as required by your project.**